# Визуализация графов. Интерактивная визуализация в Plotly и Pyvis




## Цель работы
Изучить и применить методы визуализации графовых структур для анализа финансовых транзакций и выявления мошеннических схем. Освоить работу с библиотеками Plotly и Pyvis.



## Описание набора данных
Данные моделируют систему мобильных денежных переводов (МДП). Транзакции содержат информацию об отправителе, получателе, их ролях (EU – обычный пользователь, RET – агент оператора, operator – оператор связи, Mer – поставщик услуг), сумме, типе операции (Ind – перевод между пользователями, Dt – пополнение кошелька, ArRC – оплата связи, Wl – снятие наличных, Merchant – оплата услуг), статусе и временной метке.

Мошеннические действия:
- **Кража телефона**: перевод Ind от одного EU другому EU, затем Wl от EU к RET (обналичивание).
- **Ботнет**: множество переводов ArRC от разных EU к оператору.

Необходимо визуализировать граф транзакций, выявить паттерны мошенничества и проанализировать временные характеристики.


In [1]:
!pip install pandas plotly pyvis networkx

In [2]:
import pandas as pd
import numpy as np
import networkx as nx
from pyvis.network import Network
from IPython.display import HTML, display
import plotly.graph_objects as go
import plotly.express as px

In [3]:
df_raw = pd.read_csv('FinFraud_Labelled.csv', sep='|', header=None, low_memory=False)

df = pd.DataFrame({
    'label': df_raw[0],
    'sender_id': df_raw[1],
    'receiver_id': df_raw[2],
    'amount': pd.to_numeric(df_raw[5], errors='coerce'),
    'type': df_raw[6],
    'status': df_raw[7],
    'sender_type': df_raw[23],
    'receiver_type': df_raw[24],
    'timestamp_raw': df_raw[16]
})

df = df.dropna(subset=['amount'])
df['amount'] = df['amount'].astype(float)
df['timestamp'] = pd.to_datetime(df['timestamp_raw'], format='%d/%m/%Y %H:%M:%S', errors='coerce')
df = df.dropna(subset=['timestamp'])

print(f"Загружено строк: {len(df)}\n")
print(f"Диапазон дат: {df['timestamp'].min()} – {df['timestamp'].max()}\n")
print(f"Типы транзакций:\n{df['type'].value_counts()}\n")
print(f"Роли отправителей:\n{df['sender_type'].value_counts()}\n")
print(f"Роли получателей:\n{df['receiver_type'].value_counts()}")

Загружено строк: 54848

Диапазон дат: 2011-06-01 00:09:01 – 2011-10-01 01:08:20

Типы транзакций:
type
ArRC        28312
Dt          12867
Ind          8205
Wl           5021
Merchant      443
Name: count, dtype: int64

Роли отправителей:
sender_type
EU     41981
RET    12867
Name: count, dtype: int64

Роли получателей:
receiver_type
operator    28312
EU          21072
RET          5021
MER           443
Name: count, dtype: int64


In [4]:
G = nx.DiGraph()

for _, row in df.iterrows():
    s, r = row['sender_id'], row['receiver_id']
    if s not in G:
        G.add_node(s, role=row['sender_type'])
    if r not in G:
        G.add_node(r, role=row['receiver_type'])

for _, row in df.iterrows():
    G.add_edge(row['sender_id'], row['receiver_id'],
               label=row['label'],
               amount=row['amount'],
               type=row['type'],
               status=row['status'],
               is_fraud=row['label'].startswith('F-') if pd.notna(row['label']) else False)

print(f"Узлов: {G.number_of_nodes()}, рёбер: {G.number_of_edges()}")

Узлов: 2009, рёбер: 7476


Построение графа
- Создан ориентированный граф `G` с **2009** узлами и **7476** рёбрами.
- Каждому узлу присвоена роль (EU, RET, operator, Mer).
- Каждому ребру добавлены атрибуты: сумма, тип, статус, метка мошенничества.

In [5]:
theft_chains = []
for u, v, data1 in G.edges(data=True):
    if (G.nodes[u].get('role') == 'EU' and G.nodes[v].get('role') == 'EU'
        and data1.get('type') == 'Ind'):
        for w, data2 in G[v].items():
            if (G.nodes[w].get('role') == 'RET' and data2.get('type') == 'Wl'):
                theft_chains.append((u, v, w, data1, data2))

print(f"Найдено цепочек кражи: {len(theft_chains)}")
for i, chain in enumerate(theft_chains[:5]):
    u, v, w, d1, d2 = chain
    print(f"{i+1}: {u} -> {v} ({d1['type']}, сумма={d1['amount']:.2f}) -> {w} ({d2['type']}, сумма={d2['amount']:.2f})")

Найдено цепочек кражи: 2013
1: PN_EU_3_17 -> PN_EU_0_373 (Ind, сумма=54435.88) -> PN_Ret4 (Wl, сумма=7973.74)
2: PN_EU_3_17 -> PN_EU_0_373 (Ind, сумма=54435.88) -> PN_Ret5 (Wl, сумма=15246.23)
3: PN_EU_3_17 -> PN_EU_0_373 (Ind, сумма=54435.88) -> PN_Ret6 (Wl, сумма=31851.47)
4: PN_EU_3_17 -> PN_EU_1_74 (Ind, сумма=74192.93) -> PN_Ret2 (Wl, сумма=54699.15)
5: PN_EU_3_17 -> PN_EU_1_74 (Ind, сумма=74192.93) -> PN_Ret6 (Wl, сумма=38758.42)


Выявление цепочек кражи телефона
- Найдено **2013** цепочек вида EU → EU (Ind) → RET (Wl).
- Примеры показывают крупные суммы (десятки тысяч) и последующее обналичивание.
- Такая структура соответствует паттерну кражи: перевод украденных средств на подконтрольный счёт и вывод через агента.


In [6]:
in_degree = dict(G.in_degree())
high_in = [(node, G.nodes[node].get('role', ''), deg) for node, deg in in_degree.items()
           if deg > 50 and G.nodes[node].get('role', '') in ['operator', 'RET']]
high_in.sort(key=lambda x: x[2], reverse=True)

print("Узлы с высокой входящей степенью (потенциальные центры ботнета):")
for node, role, deg in high_in[:10]:
    print(f"{node} (роль {role}): входящих рёбер {deg}")

central_node = high_in[0][0] if high_in else None
if central_node:
    print(f"\nЦентральный узел: {central_node}")
    senders = list({u for u, v in G.in_edges(central_node)})
    print(f"Количество отправителей: {len(senders)}")

Узлы с высокой входящей степенью (потенциальные центры ботнета):
operator (роль operator): входящих рёбер 1550
PN_Ret4 (роль RET): входящих рёбер 197
PN_Ret6 (роль RET): входящих рёбер 196
PN_Ret5 (роль RET): входящих рёбер 189
PN_Ret2 (роль RET): входящих рёбер 178
PN_Ret3 (роль RET): входящих рёбер 177
PN_Ret1 (роль RET): входящих рёбер 162

Центральный узел: operator
Количество отправителей: 1550


Выявление ботнета по входящей степени
- Узел operator имеет **1550** входящих рёбер (преимущественно тип ArRC).
- Это указывает на возможный ботнет: множество устройств совершают однотипные платежи оператору связи.
- RET-агенты также имеют высокую входящую степень (до 197), что соответствует обналичиванию средств.


In [7]:
color_map = {
    'EU': 'lightblue',
    'RET': 'orange',
    'operator': 'green',
    'Mer': 'purple',
    'unknown': 'gray'
}

def show_graph(graph, title, height='750px', width='100%', directed=True):
    net = Network(height=height, width=width, directed=directed, notebook=True,
                  cdn_resources='remote')
    for node, data in graph.nodes(data=True):
        role = data.get('role', 'unknown')
        color = color_map.get(role, 'gray')
        net.add_node(node, label=node[:10], title=f"Роль: {role}", color=color)
    for u, v, data in graph.edges(data=True):
        is_fraud = data.get('is_fraud', False)
        edge_color = 'red' if is_fraud else 'gray'
        title = f"Сумма: {data['amount']:.2f}\nТип: {data['type']}"
        net.add_edge(u, v, title=title, color=edge_color, width=1)
    net.show(f'{title}.html')
    display(HTML(f'{title}.html'))

Настройка визуализации Pyvis
- Определена функция show_graph, которая создаёт интерактивный HTML-граф с цветовой кодировкой узлов по ролям.
- Мошеннические рёбра выделяются красным цветом.


In [15]:
if theft_chains:
    u, v, w, d1, d2 = theft_chains[0]
    print(f"Визуализация кражи: {u} -> {v} -> {w}")
    sub_nodes = {u, v, w}
    subG = nx.DiGraph()
    for node in sub_nodes:
        subG.add_node(node, role=G.nodes[node]['role'])
    for uu, vv, data in G.edges(data=True):
        if uu in sub_nodes and vv in sub_nodes:
            subG.add_edge(uu, vv, **data)
    show_graph(subG, 'theft_example')

Визуализация кражи: PN_EU_3_17 -> PN_EU_0_373 -> PN_Ret4
Сумма: 9422.58
Тип: Dt.html


Визуализация цепочки кражи телефона
- Pyvis: интерактивный граф позволяет детально рассмотреть три узла и два ребра (Ind и Wl).
- Plotly: статическое представление с подписями узлов и цветовой дифференциацией.
- Обе визуализации наглядно демонстрируют схему: EU → EU → RET.


In [9]:
def plotly_graph(graph, title):
    pos = nx.spring_layout(graph, k=2, iterations=50)
    edge_trace = []
    for edge in graph.edges():
        x0, y0 = pos[edge[0]]
        x1, y1 = pos[edge[1]]
        edge_trace.append(go.Scatter(
            x=[x0, x1, None], y=[y0, y1, None],
            mode='lines',
            line=dict(width=1, color='gray'),
            hoverinfo='none'
        ))
    node_x = []
    node_y = []
    node_colors = []
    for node in graph.nodes():
        x, y = pos[node]
        node_x.append(x)
        node_y.append(y)
        role = graph.nodes[node].get('role', 'unknown')
        node_colors.append(color_map.get(role, 'gray'))
    node_trace = go.Scatter(
        x=node_x, y=node_y,
        mode='markers+text',
        text=[node[:10] for node in graph.nodes()],
        textposition='bottom center',
        marker=dict(size=10, color=node_colors, line=dict(width=1, color='black')),
        hoverinfo='text',
        hovertext=[f"{node}<br>Роль: {graph.nodes[node].get('role')}" for node in graph.nodes()]
    )
    fig = go.Figure(data=edge_trace + [node_trace],
                    layout=go.Layout(title=title, showlegend=False,
                                     hovermode='closest',
                                     xaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
                                     yaxis=dict(showgrid=False, zeroline=False, showticklabels=False)))
    fig.show()

if theft_chains:
    u, v, w, d1, d2 = theft_chains[0]
    sub_nodes = {u, v, w}
    subG = G.subgraph(sub_nodes).copy()
    plotly_graph(subG, 'Кража телефона (Plotly)')

In [10]:
if high_in:
    central_node, central_role, deg = high_in[0]
    print(f"Визуализация ботнета: центральный узел {central_node} (роль {central_role}), входящих рёбер {deg}")
    senders = list({u for u, v in G.in_edges(central_node)})
    if len(senders) > 30:
        senders = senders[:30]
    sub_nodes = {central_node} | set(senders)
    subG = G.subgraph(sub_nodes).copy()
    show_graph(subG, 'botnet_example')
    plotly_graph(subG, 'Ботнет (Plotly)')

Визуализация ботнета: центральный узел operator (роль operator), входящих рёбер 1550
Сумма: 3706.75
Тип: ArRC.html


Визуализация ботнета
- Центральный узел operator (роль operator) имеет 1550 входящих рёбер.
- Для наглядности построен подграф, включающий оператора и 30 отправителей.
- На графах отчётливо видна звездообразная структура: все отправители (EU) совершают однотипные транзакции ArRC в сторону оператора, что является характерным признаком ботнета.

In [11]:
df['hour'] = df['timestamp'].dt.hour
df['day'] = df['timestamp'].dt.date

fig_hour = px.histogram(df, x='hour', title='Распределение транзакций по часам суток')
fig_hour.show()

fig_day = px.histogram(df, x='day', title='Распределение транзакций по дням')
fig_day.show()

df['is_fraud'] = df['label'].str.startswith('F-', na=False)

fig_fraud_hour = px.histogram(df, x='hour', color='is_fraud',
                               title='Распределение по часам: мошеннические vs обычные',
                               barmode='group')
fig_fraud_hour.show()

- По часам суток активность распределена равномерно: минимум – 2211 транзакций (в 6 часов), максимум – 2391 транзакция (в 1 час). Значительных пиков не наблюдается.
- По дням наибольшее количество транзакций зафиксировано 15 июня 2011 года (635 операций), наименьшее – 1 октября 2011 года (26 транзакций), что связано с окончанием периода сбора данных.
- Сравнение мошеннических и обычных транзакций показывает, что мошеннические операции составляют 1,3% от общего объёма (717 из 54848). Их распределение по часам близко к равномерному, с незначительным увеличением в утренние (3–4 часа) и дневные (13 часов) периоды.


In [12]:
valid_theft = []
for u, v, w, d1, d2 in theft_chains:
    t1_series = df[(df['sender_id']==u) & (df['receiver_id']==v) & (df['type']=='Ind')]['timestamp']
    t2_series = df[(df['sender_id']==v) & (df['receiver_id']==w) & (df['type']=='Wl')]['timestamp']
    if not t1_series.empty and not t2_series.empty:
        t1 = t1_series.iloc[0]
        t2 = t2_series.iloc[0]
        if t1 < t2:
            valid_theft.append((t1, t2, (t2 - t1).total_seconds() / 60))

if valid_theft:
    deltas = [delta for _, _, delta in valid_theft]
    fig_delta = px.histogram(x=deltas, title='Разница во времени между переводом и обналичиванием (минуты)',
                             labels={'x':'минуты'})
    fig_delta.show()
    print(f"Корректных цепочек: {len(valid_theft)}")
    print(f"Средняя задержка: {np.mean(deltas):.1f} мин, медиана: {np.median(deltas):.1f} мин")
    print(f"Минимальная: {np.min(deltas):.1f} мин, максимальная: {np.max(deltas):.1f} мин")
else:
    print("Нет корректных цепочек (Ind → Wl с правильным порядком времени).")

Корректных цепочек: 563
Средняя задержка: 26760.7 мин, медиана: 12669.0 мин
Минимальная: 4.8 мин, максимальная: 164437.2 мин


- Из 2013 обнаруженных цепочек EU → EU (Ind) → RET (Wl) корректный временной порядок (перевод раньше снятия) имеют 563 цепочки.
- Средняя задержка составляет 26760,7 мин (~ 18,6 дней), медианная – 12669 мин (~ 8,8 дней).
- Минимальная задержка – всего 4,8 мин, что указывает на возможность немедленного вывода украденных средств.
- Максимальная задержка достигает 164437,2 мин (~ 114 дней), что может свидетельствовать о разных тактиках злоумышленников: от быстрого обналичивания до длительного «отмывания» средств.
- Широкий разброс значений (от минут до месяцев) подтверждает, что паттерн кражи не всегда реализуется мгновенно; задержка может быть намеренной для сокрытия следов.


In [13]:
operator_times = df[df['receiver_id'] == 'operator']['timestamp']
if not operator_times.empty:
    fig_operator_hour = px.histogram(operator_times.dt.hour, title='Входящие в operator по часам')
    fig_operator_hour.show()
    hour_counts = operator_times.dt.hour.value_counts().sort_index()
    print("Часы с наибольшей активностью:")
    print(hour_counts.head(5))

Часы с наибольшей активностью:
timestamp
0    1191
1    1219
2    1107
3    1102
4    1204
Name: count, dtype: int64


- Всего зафиксировано 28312 транзакций, направленных оператору связи.
- Активность распределена по часам суток равномерно: минимальное значение – 1102 транзакции (3 часа), максимальное – 1273 транзакции (7 часов). Разница между пиковым и минимальным часом составляет всего 171 транзакцию (~13%).
- Отсутствие ярко выраженного ночного спада и равномерная нагрузка на протяжении всех 24 часов свидетельствуют об автоматизированном характере платежей, что характерно для ботнета, где заражённые устройства совершают транзакции независимо от времени суток.

In [14]:
print("--- Статистика всего графа ---")
print(f"Количество узлов: {G.number_of_nodes()}")
print(f"Количество рёбер: {G.number_of_edges()}")
degrees = [d for n, d in G.degree()]
print(f"Средняя степень (входящая+исходящая): {np.mean(degrees):.2f}")
print(f"Медианная степень: {np.median(degrees):.2f}")

G_und = G.to_undirected()
clustering = nx.clustering(G_und)
avg_clustering = np.mean(list(clustering.values()))
print(f"Средний коэффициент кластеризации: {avg_clustering:.4f}")

fraud_edges = [(u, v) for u, v, d in G.edges(data=True) if d.get('is_fraud', False)]
fraud_sub = G.edge_subgraph(fraud_edges).copy()
print(f"\n--- Мошеннические рёбра ---")
print(f"Количество узлов: {fraud_sub.number_of_nodes()}, рёбер: {fraud_sub.number_of_edges()}")
if fraud_sub.number_of_nodes() > 0:
    fraud_degrees = [d for n, d in fraud_sub.degree()]
    print(f"Средняя степень в мошенническом подграфе: {np.mean(fraud_degrees):.2f}")

normal_edges = [(u, v) for u, v, d in G.edges(data=True) if not d.get('is_fraud', False)]
normal_sub = G.edge_subgraph(normal_edges).copy()
print(f"\n--- Обычные рёбра ---")
print(f"Количество узлов: {normal_sub.number_of_nodes()}, рёбер: {normal_sub.number_of_edges()}")
if normal_sub.number_of_nodes() > 0:
    normal_degrees = [d for n, d in normal_sub.degree()]
    print(f"Средняя степень в обычном подграфе: {np.mean(normal_degrees):.2f}")

--- Статистика всего графа ---
Количество узлов: 2009
Количество рёбер: 7476
Средняя степень (входящая+исходящая): 7.44
Медианная степень: 3.00
Средний коэффициент кластеризации: 0.1681

--- Мошеннические рёбра ---
Количество узлов: 10, рёбер: 24
Средняя степень в мошенническом подграфе: 4.80

--- Обычные рёбра ---
Количество узлов: 2009, рёбер: 7452
Средняя степень в обычном подграфе: 7.42


- Общая структура: граф содержит 2009 узлов и 7476 рёбер. Средняя степень узла составляет 7,44, медианная – 3,00. Существенный разрыв между средним и медианным значениями указывает на наличие узлов с высокой степенью (хабов) при большом количестве узлов с низкой связностью.

- Коэффициент кластеризации: среднее значение 0,1681 – низкое, что характерно для сетей с выраженными звёздообразными структурами (ботнет) и отсутствием плотных сообществ.

- Мошеннический подграф: включает 10 узлов и 24 ребра. Средняя степень в нём составляет 4,80, что выше медианной степени всего графа (3,00). Это подтверждает, что мошеннические транзакции концентрируются вокруг небольшого числа узлов, образуя более плотные связи.

- Обычный подграф: охватывает все 2009 узлов и 7452 ребра, средняя степень – 7,42. Значение близко к общей средней степени, что ожидаемо, так как мошеннические рёбра составляют лишь 0,3% от общего числа.

- Полученные статистические характеристики согласуются с визуальными наблюдениями: наличие хабов (оператор, RET-агенты) обусловливает высокую среднюю степень при низкой медиане, а низкий коэффициент кластеризации подтверждает звездообразную структуру ботнета.